# Test Generator

This notebook tests `rag/generator.py`.

The first test uses a fake OpenAI client, so it does not spend API tokens.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "rag").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

## 1. Test answer generation with a fake OpenAI client

This checks our generator logic without calling the OpenAI API.

In [ ]:
from rag.generator import generate_answer


class FakeResponse:
    output_text = "The project is a RAG prototype for answering questions from uploaded documents. [Source 1]"


class FakeResponses:
    def create(self, **kwargs):
        print("Prompt sent to fake OpenAI client:\n")
        print(kwargs["input"])
        return FakeResponse()


class FakeOpenAIClient:
    responses = FakeResponses()


retrieved_chunks = [
    {
        "text": "Ask My Documents is a Retrieval-Augmented Generation prototype for uploaded documents.",
        "citation": "Project_Proposal.docx, Paragraph 2",
        "document_name": "Project_Proposal.docx",
        "source_label": "Paragraph 2",
        "score": 0.88,
    }
]

result = generate_answer(
    question="What is this project about?",
    retrieved_chunks=retrieved_chunks,
    model="fake-model",
    client=FakeOpenAIClient(),
)

result

## 2. Test abstention when there are no retrieved chunks

If retrieval finds no useful evidence, the generator should not call OpenAI.

In [ ]:
empty_result = generate_answer(
    question="What is not in the documents?",
    retrieved_chunks=[],
    model="fake-model",
    client=FakeOpenAIClient(),
)

empty_result

## 3. Optional: call the real OpenAI API

Only run this after adding a real `OPENAI_API_KEY` and `OPENAI_CHAT_MODEL` to `.env`.

In [ ]:
RUN_REAL_OPENAI_CALL = False

if RUN_REAL_OPENAI_CALL:
    real_result = generate_answer(
        question="What is this project about?",
        retrieved_chunks=retrieved_chunks,
    )
    real_result
else:
    print("Real OpenAI call skipped. Set RUN_REAL_OPENAI_CALL = True when ready.")